In [1]:
import pandas as pd
import numpy as np
import re
import string
import pickle

In [2]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text=text.replace(punctuation,'')
    return text

In [3]:
with open('../static/model/corpora/stopwords/english','r') as file:
    sw=file.read().splitlines()

In [4]:
from nltk.stem import PorterStemmer
ps=PorterStemmer()

In [13]:
def preprocessing(text):
    data=pd.DataFrame([text],columns=["tweet"])
    
    data["tweet"]=data["tweet"].apply(lambda x: " ".join(x.lower() for x in x.split()))
    data["tweet"]=data["tweet"].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*','',x,flags=re.MULTILINE) for x in x.split()))
    data["tweet"]=data["tweet"].apply(remove_punctuations)
    data["tweet"]=data["tweet"].str.replace('\d+','',regex=True)
    data["tweet"]=data["tweet"].apply(lambda x: " ".join(x for x in x.split() if x not in sw))
    data["tweet"]=data["tweet"].apply(lambda x: " ".join(ps.stem(x) for x in x.split() ))

    return data["tweet"].tolist()


In [15]:
txt='bad product .I hate it'

In [16]:
txt

'bad product .I hate it'

In [17]:
preprocessed_txt=preprocessing(txt)

In [18]:
preprocessed_txt

['bad product hate']

In [ ]:
vocab=pd.read_csv('../static/model/vocabulary.txt',header=None)
tokens=vocab[0].tolist()

In [11]:
tokens

['test',
 'android',
 'app',
 'beauti',
 'cute',
 'health',
 'iger',
 'iphoneonli',
 'iphonesia',
 'iphon',
 'final',
 'case',
 'thank',
 'yay',
 'soni',
 'xperia',
 'love',
 'would',
 'go',
 'talk',
 'relax',
 'smartphon',
 'wifi',
 'connect',
 'im',
 'know',
 'made',
 'way',
 'home',
 'amaz',
 'servic',
 'appl',
 'wont',
 'even',
 'question',
 'pay',
 'stupid',
 'support',
 'softwar',
 'updat',
 'fuck',
 'phone',
 'big',
 'time',
 'happi',
 'us',
 'instap',
 'instadaili',
 'xperiaz',
 'new',
 'type',
 'c',
 'charger',
 'cabl',
 'uk',
 '…',
 'amazon',
 'year',
 'newyear',
 'start',
 'technolog',
 'samsunggalaxi',
 'iphonex',
 'shop',
 'listen',
 'music',
 'likeforlik',
 'photo',
 'fun',
 'selfi',
 'water',
 'camera',
 'picoftheday',
 'sun',
 'instagood',
 'boy',
 'outdoor',
 'hey',
 'make',
 'ipod',
 'dont',
 'color',
 'inch',
 'crash',
 'everi',
 'need',
 'realli',
 'drop',
 'ball',
 'design',
 'give',
 'anoth',
 'crazi',
 'purchas',
 'lol',
 'work',
 'hard',
 'play',
 'ipad',
 'batt

In [19]:
def vectorizer(ds,vocabulary):
    vectorized_list=[]
    for sentence in ds:
        sentence_list=np.zeros(len(vocabulary))

        for i in range(len(vocabulary)):
            if vocabulary[i] in sentence.split():
                sentence_list[i]=1

        vectorized_list.append(sentence_list)

    vectorized_list_new=np.asarray(vectorized_list,dtype=np.float32)
    return vectorized_list_new

In [20]:
vectorized_txt=vectorizer(preprocessed_txt,tokens)

In [21]:
with open('../static/model/model.pickle','rb') as f:
    model=pickle.load(f)

In [22]:
def get_prediction(vectorized_txt):
    prediction=model.predict(vectorized_txt)
    if prediction==1:
        return 'negative'
    else:
        return 'positive'

In [23]:
txt="awesome product.I love."
preprocessed_txt=preprocessing(txt)
vectorized_txt=vectorizer(preprocessed_txt,tokens)
prediction=get_prediction(vectorized_txt)
prediction


'positive'